In [ ]:
def load_state(state):
    state['user_choice']['null_columns'] = {
    "null_columns": {"drop_column": ["Cabin"],
    "drop_rows": ["Embarked"],
    "fill_with_average": ["Age"]}}

    state['user_choice']["metrics"] = {
    "metrics": {"True": ["f1", "accuracy", "roc_auc", "confusion_matrix"]}}

    state['user_choice']["models"] = {
    "models": {"True": ["logistic_regression", "random_forest", "xgboost"]}}

    state['user_choice']["encoding"] = {
        'one-hot': ['Sex', 'Embarked'],
        'ordinal-encoding': [],
        'binary-encoding': ['Name', 'Cabin'],
        'frequency-encoding': [],
        'target-encoding': ['Ticket']}

    state['user_choice']["skew"] = {
    "skew": {"skewed": ["SibSp", "Parch", "Fare"]}}

    state['user_choice']["scaling"] = {
    "scaling": {"scaling": {"RobustScaler": ["logistic_regression", "random_forest", "xgboost"]}}}

    state['user_choice']["hp_choice"] = {
    "hp_choice": {"True": {"n_estimators": 10, "max_depth": 1, "min_samples_split": 2, "min_samples_leaf": 1, "max_features": "auto", "bootstrap": True, "criterion": "gini"}}}

    state['user_choice']["final"] = {
    "final": {"final": {"models": "xgboost", "scaling": "standard_scaler", "imbalance_methods": "random_undersampling"}}}

    state['user_choice']["sampling"] = {
    "sampling": {"sampling": True}}
    return state

In [1]:
from state import MessagesState
from langgraph.types import interrupt
from langgraph.graph import StateGraph, START, END
from langchain.messages import HumanMessage, SystemMessage
from config.config_dicts.null_handler import null_handler_config
from config.model import *

def saver(state):
    import pickle
    with open("pickles/encoding_results.pkl", "wb") as f:
        pickle.dump(state, f)
    return state

def build_hitl_subgraph(config: dict, orderlist: list):
    node_dict= {"analysis": 0, "struct": 0, "interrupt": 0, "update": 0}
    numbered_orderlist = []
    for node in orderlist:
        numbered_orderlist.append(node+str(node_dict[node]))
        node_dict[node]+=1
    def struct_node(state):
        # import pickle
        # with open("pickles/state.pkl", "rb") as f:
        #     loaded_state = pickle.load(f)

        node_count = int(numbered_orderlist[state['subgraph']][-1])
        current_struct = config['struct'][node_count] 
      
        model = current_struct['model']
        struct_model = model.with_structured_output(current_struct["schema"])

        message_template = current_struct["prompt"]
        context_demands = current_struct.get("context_demands",{})
        if context_demands != {}:
            prompt_string = ""
            for key,value in context_demands.items():
                prompt_string += str(state[key][value]) + "\n\n"
            human_message = message_template.format(context = prompt_string)
        else:
            human_message = message_template
        messages = [SystemMessage(content=prompt_generator(state, current_struct.get("data_demands",[]))), human_message]
        # return {"subgraph": 1, "struct": {"structresponse": "value"}}
        options_dict = struct_model.invoke(messages)
        struct_dict = state['struct']
        struct_dict[current_struct["output_name"]] = options_dict.model_dump_json()
        # struct_dict[current_struct["output_name"]] = {"teststrudtnode": "value tsetlsekjl"}

        # try:
        #     import pickle
        #     with open("pickles/struct.pkl", "wb") as f:
        #         pickle.dump(state, f)
        # except Exception as e:
        #     print(f"Error at {e}")
        # return {"struct": options_dict.model_dump_json()}
        
        return {
            "struct": struct_dict,
            "subgraph": state["subgraph"] + 1,
        }
        # else:
        #     return {
        #         "struct": struct_dict,
        #         "subgraph": 1
        #         }
        



    def ask_user_node(state):

        # try:
        #     import pickle
        #     with open("pickles/struct.pkl", "wb") as f:
        #         pickle.dump(state, f)
        # except Exception as e:
        #     print(f"Error at {e}")
        if config.get("interrupt_message", None):
            interrupt_message = config["interrupt_message"]
            n_rows = state['df_info']['n_rows']
            n_rows_20 = int(n_rows * 0.2)
            n_rows_10 = int(n_rows * 0.1)
            user_choice = interrupt({"interrupt_message": interrupt_message.format(n_rows=n_rows, n_rows_20=n_rows_20, n_rows_10=n_rows_10)})
        else:
            user_choice = interrupt({"struct": state.get("struct",{})})
        state_dict = state['user_choice']
        node_count = int(numbered_orderlist[state['subgraph']][-1])
        state_dict[config["interrupt_name"][node_count]] = user_choice
        return {
            "user_choice": state_dict,
            "subgraph": state["subgraph"] + 1,
        }
              
    graph_builder = StateGraph(MessagesState)
    node_dict = {
        "struct": struct_node,
        "interrupt": ask_user_node,
        "update": config.get("update", {}).get("method", None)
    }

    for i,node in enumerate(orderlist):
        graph_builder.add_node(numbered_orderlist[i],node_dict[node])

    for i,node in enumerate(numbered_orderlist):
        graph_builder.add_edge(START, node) if i == 0 else graph_builder.add_edge(numbered_orderlist[i-1],node)
    graph_builder.add_edge(numbered_orderlist[-1],END)

    graph = graph_builder.compile()
    return graph

from analyst.nodes import analyst_call
# from config.config_dicts.null_handler import null_handler_config
# from config.config_dicts.metric_chooser import metrics_config
# from config.config_dicts.model_chooser import models_config
from config.config_dicts.encoding_handler import encoding_config
# from config.config_dicts.skew_handler import skew_config
# from config.config_dicts.sampling_query import sampling_config
from config.config_dicts.hp_chooser import hp_config
# from config.nodes import preprocessor_call
from state import *
from langgraph.graph import StateGraph, START, END
from analyst.tools import tools_by_name as analyst_tools
from langgraph.checkpoint.memory import InMemorySaver

agent_builder = StateGraph(MessagesState)
analyst_tool_node = make_tool_node(analyst_tools)
null_subgraph = build_hitl_subgraph(encoding_config, ["struct",'struct','interrupt'])
# agent_builder.add_node("analyst", analyst_call)
# agent_builder.add_node("analyst_tools", analyst_tool_node)
# agent_builder.add_node("test_node", load_state)
agent_builder.add_node("load_df",load_df)
agent_builder.add_node("null_subgraph", null_subgraph)
agent_builder.add_node("saver", saver)
# Add edges to connect nodes
# TEMPORARY
agent_builder.add_edge(START, "load_df")
agent_builder.add_edge("load_df", "null_subgraph")
# agent_builder.add_conditional_edges(
#     "analyst",
#     should_continue,
#     ["analyst_tools", "saver"]
# )
agent_builder.add_edge("null_subgraph", "saver")
agent_builder.add_edge("saver", END)

checkpointer = InMemorySaver()
agent = agent_builder.compile(checkpointer=checkpointer)

config = {
    "configurable": {
        "thread_id": "chat-1",
    }
}
from langchain.messages import HumanMessage, SystemMessage
messages = [SystemMessage(content="start your task"),HumanMessage(content="Analyze the data")]

# Use stream() instead of invoke() to enable input() functionality
for chunk in agent.stream({"messages": messages, "subgraph": 0, "user_choice": {}, "struct": {}, "df_info": {"target": "Survived", "filepath": "datasets/Titanic-Dataset.csv", "data_description": "Test"}}, config): 
    # Process each chunk as it arrives
    for node_name, node_output in chunk.items():
        print(f"\n[{node_name}]")
        if node_name == "__interrupt__":
            interrupt_payload = node_output
            break
        if "messages" in node_output and len(node_output["messages"]) > 0:
            last_message = node_output["messages"][-1]
            if hasattr(last_message, "content") and last_message.content:
                print(f"Content: {last_message.content}")
            if hasattr(last_message, "tool_calls") and last_message.tool_calls:
                print(f"Tool calls: {last_message.tool_calls}")
        else:
            print(f"Output: {node_output}")


[load_df]
Output: {'df_info': {'target': 'Survived', 'filepath': 'datasets/Titanic-Dataset.csv', 'data_description': 'Test', 'n_rows': 891, 'n_cols': 12, 'numeric_features': {'count': 7, 'data': ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']}, 'categorical_features': {'count': 5, 'data': ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']}, 'numeric_stats': {'PassengerId': {'type': 'numeric', 'mean': 446.0, 'std': 257.3538420152301, 'min': 1.0, 'max': 891.0, 'skew': 0.0, 'outlier_count': 0}, 'Survived': {'type': 'numeric', 'mean': 0.3838, 'std': 0.4865924542648575, 'min': 0.0, 'max': 1.0, 'skew': 0.4785234382949897, 'outlier_count': 0}, 'Pclass': {'type': 'numeric', 'mean': 2.3086, 'std': 0.836071240977049, 'min': 1.0, 'max': 3.0, 'skew': -0.6305479068752845, 'outlier_count': 0}, 'Age': {'type': 'numeric', 'mean': 29.6991, 'std': 14.526497332334042, 'min': 0.42, 'max': 80.0, 'skew': 0.38910778230082704, 'outlier_count': 11}, 'SibSp': {'type': 'numeric', 'mean': 0.52

In [2]:
import json
json.loads(interrupt_payload[0].value['struct']['enc0'])['actions']

{'one-hot': ['Sex', 'Embarked'],
 'ordinal-encoding': [],
 'binary-encoding': ['Name', 'Cabin'],
 'frequency-encoding': [],
 'target-encoding': ['Ticket']}

In [7]:
import json
print(json.loads(interrupt_payload[0].value))

TypeError: the JSON object must be str, bytes or bytearray, not dict

In [3]:
import json
print(json.loads(interrupt_payload[0].value['struct']['sca0'])["actions"])

{'StandardScaler': [], 'RobustScaler': ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare'], 'no_scaling': []}


In [ ]:
import pickle
with open("pickles/encoding_handling.pkl", "rb") as f:
    loaded_state = pickle.load(f)

loaded_state

In [2]:
# Stream (required for interrupt to work)
from langchain.messages import HumanMessage, SystemMessage
messages = [SystemMessage(content="start your task"),HumanMessage(content="Analyze the data")]

# Use stream() instead of invoke() to enable input() functionality
for chunk in agent.stream({"messages": messages, "subgraph": 0, "user_choice": {}, "struct": {}, "df_info": {"target": "Survived", "filepath": "datasets/Titanic-Dataset.csv", "data_description": ""}}, config): 
    # Process each chunk as it arrives
    for node_name, node_output in chunk.items():
        print(f"\n[{node_name}]")
        if node_name == "__interrupt__":
            interrupt_payload = node_output
            break
        if "messages" in node_output and len(node_output["messages"]) > 0:
            last_message = node_output["messages"][-1]
            if hasattr(last_message, "content") and last_message.content:
                print(f"Content: {last_message.content}")
            if hasattr(last_message, "tool_calls") and last_message.tool_calls:
                print(f"Tool calls: {last_message.tool_calls}")
        else:
            print(f"Output: {node_output}")


[load_df]
Output: {'df_info': {'target': 'Survived', 'filepath': 'datasets/Titanic-Dataset.csv', 'data_description': '', 'n_rows': 891, 'n_cols': 12, 'numeric_features': {'count': 7, 'data': Index(['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare'], dtype='object')}, 'categorical_features': {'count': 5, 'data': Index(['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked'], dtype='object')}, 'numeric_stats': {'PassengerId': {'type': 'numeric', 'mean': np.float64(446.0), 'std': 257.3538420152301, 'min': 1.0, 'max': 891.0, 'skew': 0.0, 'outlier_count': 0}, 'Survived': {'type': 'numeric', 'mean': np.float64(0.3838), 'std': 0.4865924542648575, 'min': 0.0, 'max': 1.0, 'skew': 0.4785234382949897, 'outlier_count': 0}, 'Pclass': {'type': 'numeric', 'mean': np.float64(2.3086), 'std': 0.836071240977049, 'min': 1.0, 'max': 3.0, 'skew': -0.6305479068752845, 'outlier_count': 0}, 'Age': {'type': 'numeric', 'mean': np.float64(29.6991), 'std': 14.526497332334042, 'min': 0.42, 'max': 80.0, 's

KeyError: 'Sample Data'

In [3]:
from langgraph.types import Command

# config = {"configurable": {"thread_id": "chat-1"}}

# # First run: hits interrupt and pauses
# result = agent.invoke({"input": "hi"}, config=config)
# print(result["__interrupt__"])

# Resume: this continues, and interrupt() returns this value

agent.invoke(Command(resume=

{
    # "null_columns": {"drop_column": ["Cabin"],
    # "drop_rows": ["Embarked"],
    # "fill_with_average": ["Age"]}

    # "metrics": {"True": ["f1", "recall", "precision"]}

    # "models": {'True': ['svm', 'random_forest', 'lightgbm']}

    "encoding": {'one-hot': ['Sex', 'Embarked'],
    'ordinal-encoding': [],
    'binary-encoding': ['Name', 'Cabin'],
    'frequency-encoding': [],
    'target-encoding': ['Ticket']}

    # "skew": {'skewed': ['SibSp', 'Parch', 'Fare']}
}

),config=config)

{'messages': [SystemMessage(content='start your task', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Analyze the data', additional_kwargs={}, response_metadata={}),
  SystemMessage(content='start your task', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Analyze the data', additional_kwargs={}, response_metadata={}),
  SystemMessage(content='start your task', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Analyze the data', additional_kwargs={}, response_metadata={}),
  SystemMessage(content='start your task', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Analyze the data', additional_kwargs={}, response_metadata={})],
 'struct': {'enc0': '{"reasoning":"I have examined the categorical columns in the dataset and recommended encoding methods based on their cardinality, nature (nominal vs. ordinal), and potential for feature utility. The primary goal is to represent categorical data numerically while

In [8]:
import json
json.loads(interrupt_payload[0].value['struct']['met0'])

{'reasoning': 'The dataset contains information about Titanic passengers, including whether they survived or not. This is a binary classification problem, where the goal is to predict survival. Accuracy is a common metric for binary classification, but it can be misleading if the classes are imbalanced. Precision and recall are useful when the cost of false positives or false negatives, respectively, is high. ROC AUC is a good metric for evaluating the overall performance of a classifier across different thresholds. Given the potential for class imbalance (more non-survivors than survivors based on the class distribution), ROC AUC, precision, and recall are strong candidates for evaluation. Accuracy can also be considered, but with a caution about class imbalance.',
 'actions': {'True': ['roc-auc', 'precision', 'recall'],
  'False': ['accuracy', 'f1']},
 'prompts': ["For this binary classification task predicting passenger survival, I recommend using ROC AUC, precision, or recall as ev